In [13]:
%load_ext autoreload
%autoreload 2
import dask
import dask.distributed
from dask_util import DaskClient
import dask_util
import numpy as np

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
local_params = {
    "n_workers":4, 
    "processes" : True, 
    "dashboard_address" : 'localhost:7777'
    
}

# cluster = {
#     "cores" : 24,
#     "processes" : 1,
#     "memory" : "1GB",
#     "shebang" : '#!/usr/bin/env bash',
#     "queue" : "serc",
#     "walltime" : "00:10:00",
#     "local_directory" : '/tmp',
#     "death_timeout" : "15s",
#     "interface" : "ib0",
#     "log_directory" : f'{os.environ["SCRATCH"]}/dask_jobqueue_logs/'    
# }


client = DaskClient(local_params=local_params)

/usr/local/lib/python3.9/dist-packages/distributed/node.py:182: UserWarning: Port 7777 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 34773 instead
  warnings.warn(


In [15]:
client.getWorkerIds()

['tcp://127.0.0.1:34421',
 'tcp://127.0.0.1:40311',
 'tcp://127.0.0.1:41727',
 'tcp://127.0.0.1:45973']

In [16]:
%load_ext autoreload
%autoreload 2
import SepVector
from __pyDaskVector import DaskVector
from __pyDaskOperator import DaskOperator
import Hypercube
import pyOperator as Op


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
WARNING! DATAPATH not found. The folder /tmp will be used to write binary files


/usr/local/lib/python3.9/dist-packages/numpy/core/getlimits.py:500: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/lib/python3.9/dist-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/usr/local/lib/python3.9/dist-packages/numpy/core/getlimits.py:500: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/lib/python3.9/dist-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  return self._float_to_str(self.smallest_subnormal)


In [30]:

ns = [10,10,10]
os = [0,0,0]
ds = [1,1,1]
chunks = (1,1,3)

ax = Hypercube.axis(n=1, o=1, d=1)
hyp = Hypercube.hypercube(ns=ns, ds=ds, os=os)
vec = SepVector.getSepVector(ns=ns, ds=ds, os=os)
vec.set(1)

floatVector
Axis 1: n=10	o=0.000000	d=1.000000
Axis 2: n=10	o=0.000000	d=1.000000
Axis 3: n=10	o=0.000000	d=1.000000

In [31]:
# option 1
# creating from scratch
data = DaskVector(client, vecCls=SepVector.floatVector, ns=ns, ds=ds, os=os, chunks=chunks)

In [32]:
# option 2
# creating from existing in-memory SepVector
daskVec = DaskVector(client, from_vector=vec, chunks=chunks)

In [33]:
client.getClient().has_what()

{'tcp://127.0.0.1:34421': ('window-bccdd4dba2df460189f63b64b26eb133',
  'window-071b0a90614ad5509c70aec81e9a64c1'),
 'tcp://127.0.0.1:40311': ('window-b67e9fdc0850d5e4409bbee363147ff0',
  'floatVector-7d8d42da18c0140433331a6976fb5dd2',
  'window-adec077dbb939d9b809e78b0caeac190'),
 'tcp://127.0.0.1:41727': ('window-182b94a63e198d31d230b628de6c2b1a',
  'floatVector-2a2c2f62386295b2febfbb5ae630ed9a'),
 'tcp://127.0.0.1:45973': ('window-49dd2856d64078c4c0d5cac932469b47',
  'floatVector-c1b0bb503e39a722b4581617fe992cbc',
  'window-84405a13c3d4cd80e75645c3264d25f6')}

In [39]:
daskVec.fut

[<Future: finished, type: SepVector.floatVector, key: window-071b0a90614ad5509c70aec81e9a64c1>,
 <Future: finished, type: SepVector.floatVector, key: window-adec077dbb939d9b809e78b0caeac190>,
 <Future: finished, type: SepVector.floatVector, key: window-84405a13c3d4cd80e75645c3264d25f6>]

In [42]:
daskVec[2,:,:]

[array([[[1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.]]], dtype=float32),
 array([], shape=(1, 0, 3), dtype=float32),
 array([[[1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.]]], dtype=float32)]

In [11]:
spread_op = DaskSpread(client, data)

In [12]:
spread_op.workers

{('tcp://127.0.0.1:34237',),
 ('tcp://127.0.0.1:38923',),
 ('tcp://127.0.0.1:43503',),
 ('tcp://127.0.0.1:45537',)}

In [14]:
scaleOp = DaskOperator(client, opCls=Op.scalingOp, domain=daskVec, range=data, op_args=[4])

In [15]:
data[:]

[array([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]], dtype=float32),
 array([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]], dtype=float32),
 array([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]], dtype=float32),
 array([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]], dtype=float32)]

In [16]:
scaleOp.forward(False, daskVec, data)

In [17]:
data[:]

/usr/local/lib/python3.9/dist-packages/numpy/core/getlimits.py:500: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/lib/python3.9/dist-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/usr/local/lib/python3.9/dist-packages/numpy/core/getlimits.py:500: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/lib/python3.9/dist-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/usr/local/lib/python3.9/dist-packages/numpy/core/getlimits.py:500: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is z

[array([[4., 4., 4., 4., 4.],
        [4., 4., 4., 4., 4.],
        [4., 4., 4., 4., 4.],
        [4., 4., 4., 4., 4.],
        [4., 4., 4., 4., 4.]], dtype=float32),
 array([[4., 4., 4., 4., 4.],
        [4., 4., 4., 4., 4.],
        [4., 4., 4., 4., 4.],
        [4., 4., 4., 4., 4.],
        [4., 4., 4., 4., 4.]], dtype=float32),
 array([[4., 4., 4., 4., 4.],
        [4., 4., 4., 4., 4.],
        [4., 4., 4., 4., 4.],
        [4., 4., 4., 4., 4.],
        [4., 4., 4., 4., 4.]], dtype=float32),
 array([[4., 4., 4., 4., 4.],
        [4., 4., 4., 4., 4.],
        [4., 4., 4., 4., 4.],
        [4., 4., 4., 4., 4.],
        [4., 4., 4., 4., 4.]], dtype=float32)]

In [24]:
scaleOp.adjoint(False, daskVec, data)

In [25]:
daskVec[:]

[array([[16., 16., 16., 16., 16.],
        [16., 16., 16., 16., 16.],
        [16., 16., 16., 16., 16.],
        [16., 16., 16., 16., 16.],
        [16., 16., 16., 16., 16.]], dtype=float32),
 array([[16., 16., 16., 16., 16.],
        [16., 16., 16., 16., 16.],
        [16., 16., 16., 16., 16.],
        [16., 16., 16., 16., 16.],
        [16., 16., 16., 16., 16.]], dtype=float32),
 array([[16., 16., 16., 16., 16.],
        [16., 16., 16., 16., 16.],
        [16., 16., 16., 16., 16.],
        [16., 16., 16., 16., 16.],
        [16., 16., 16., 16., 16.]], dtype=float32),
 array([[16., 16., 16., 16., 16.],
        [16., 16., 16., 16., 16.],
        [16., 16., 16., 16., 16.],
        [16., 16., 16., 16., 16.],
        [16., 16., 16., 16., 16.]], dtype=float32)]